# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their `@id`s, and contained fields (with their `@id`s).

The Croissant metadata provides a list of data `recordSet` objects, each with associated fields. We enumerate them here:

In [ ]:
from pprint import pprint

# Show each record set, its @id, and its fields (by @id)
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        print(f"RecordSet: @id={rs['@id']}, name={rs.get('name', 'N/A')}")
        fields = rs.get('field', [])
        # field might be a dict if there's only one, so coerce to list
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("  Fields:")
            for f in fields:
                fid = f['@id'] if isinstance(f, dict) else f
                print(f"    - {fid}")
        else:
            print("  (No fields listed)")
        print()
else:
    print("No record sets found in this dataset metadata.")

Let's programmatically collect available record set `@id`s for further analysis.

In [ ]:
# Gather list of record set IDs
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        record_sets.append(rs['@id'])

print(f"Record set @ids: {record_sets}")

Optionally, display a sample record for each available record set using its `@id` as reference.

In [ ]:
# If any record sets found, print a sample record from each
for rsid in record_sets:
    print(f"Sample records from RecordSet {rsid}:\n")
    try:
        for i, rec in enumerate(dataset.records(record_set=rsid)):
            pprint(rec)
            if i>=0: # Show just the first record
                break
    except Exception as e:
        print(f"Could not read records for {rsid}: {e}")
    print("\n-------------------\n")

## 3. Data Extraction
Load data from the available record sets into DataFrames for analysis.

You can reference particular fields (columns) by their `@id`s as shown above.

In [ ]:
# Extract data from all found record sets into dataframes (if any discovered)
dataframes = {}

for rsid in record_sets:
    print(f"Loading records for {rsid} ...")
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"  Could not load records for {rsid}: {e}")
print(f"Loaded dataframes for {list(dataframes.keys())}")
# For subsequent analysis, you may select a record set by @id as shown below:
if dataframes:
    some_record_set_id = list(dataframes.keys())[0] # for demonstration, pick the first
    print(f"Using record set: {some_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- EDA Preparation ---

# Pick a DataFrame (record set) to analyze
if dataframes:
    record_set_id = list(dataframes.keys())[0] # choose first loaded record set
    df = dataframes[record_set_id]
    print(f"Working with RecordSet: {record_set_id}")
    
    # Show all available columns
    print("Available columns (fields, by @id or label):")
    print(df.columns.tolist())
    
    # Try to guess a numeric field, e.g. 'Molecular Weight' or 'MW' or one with float/integer data
    # We simply pick the first float/int column, if available
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try parsing fields as float to find a numeric field
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except:
                continue
    print(f"Selected numeric_field: {numeric_field}")
    
    if numeric_field:
        threshold = df[numeric_field].mean()  # demo: filter above mean
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a categorical field if available
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() > 1 and df[col].nunique() < 32:
                group_field = col
                break
        print(f"Selected group_field: {group_field}")
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
else:
    print("No dataframes were loaded. Please check data extraction section above.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic visualization of numeric field distribution, if data available
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], kde=True, bins=40)
    plt.title(f"Distribution of {numeric_field} in RecordSet {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If filtered and grouped, show a bar plot
    if 'filtered_df' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        grouped_df.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field} (Filtered)")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The dataset provides mass spectrometry-derived protein records and associated features extracted from human extracellular vesicles isolated from mast cells. 
* We demonstrated how to programmatically identify available record sets and fields by `@id`, and how to use `mlcroissant` to query, filter, normalize, and visualize the data.
* Such an approach can be extended for feature engineering, more advanced visualizations, or downstream machine learning as needed.

Refer to the [FAIR^2 Croissant documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for details on schema and record definitions.